In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from matplotlib.colors import LinearSegmentedColormap


def read_fastani_output(file_path):
    """read FastANI output file"""
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                query = parts[0].replace('_query', '').replace('_fna', '').replace('_fa', '').replace('_fasta', '')
                query = query.replace('_order_qry_fasta', '').replace('_v1', '')
                ref = parts[1].replace('_ref', '').replace('_fna', '').replace('_fa', '').replace('_fasta', '')
                ref = ref.replace('_order_qry_fasta', '').replace('_v1', '')
                ani_value = float(parts[2])
                data.append({'query': query, 'ref': ref, 'ani': ani_value})
    return pd.DataFrame(data)


def build_similarity_matrix(df):
    """generate a similariry matrix """
    samples = sorted(set(df['query'].tolist() + df['ref'].tolist()))

    matrix = pd.DataFrame(np.full((len(samples), len(samples)), np.nan), 
                          index=samples, columns=samples) 

    for _, row in df.iterrows(): 
        q, r, ani = row['query'], row['ref'], row['ani']
        matrix.loc[q, r] = ani

    for i in samples:
        for j in samples:
            if pd.isna(matrix.loc[i, j]) and not pd.isna(matrix.loc[j, i]): 
                matrix.loc[i, j] = matrix.loc[j, i]
        matrix.loc[i, i] = 100.0


    matrix = matrix.astype(float) 
    matrix = matrix.fillna(matrix.min().min()) 
    
    return matrix


def plot_clustered_heatmap(matrix, title="FastANI Clustered Heatmap",
                           output_file=None, figsize=(14, 14)):
    """plot heat map"""
    plt.rcParams['axes.unicode_minus'] = False

    distance = 1 - matrix.values / 100.0
    distance = (distance + distance.T) / 2.0
    np.fill_diagonal(distance, 0.0) 
    condensed = squareform(distance, checks=False) 

    link = linkage(condensed, method='average')


    colors = ['#d73027', '#f46d43', '#fdae61', '#fee08b',
              '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850']
    cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=256)

    g = sns.clustermap(matrix,
                       row_linkage=link,
                       col_linkage=link,
                       cmap=cmap,
                       vmin=75,
                       vmax=100,
                       annot=True,
                       fmt='.1f',
                       annot_kws={'size': 7},
                       linewidths=0.5,
                       figsize=figsize,
                       cbar_kws={'label': 'ANI (%)', 'shrink': 0.8},
                       cbar_pos=(0.9, 0.9, 0.02, 0.1), ## (x, y, width, height)
                       dendrogram_ratio=0.1,
                       colors_ratio=0.03)

    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(),rotation=45, ha='right', fontsize=9)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(),rotation=0, fontsize=9)
    g.ax_heatmap.set_xlabel('Reference', fontsize=12)
    g.ax_heatmap.set_ylabel('Query', fontsize=12)
    g.fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    g.savefig(output_file, dpi=300, bbox_inches='tight')
    
    return g


def main():
    root = '/home/octopus/planet/claviceps'
    file_path = f'{root}/MBE/input/allClaviceps.FastANI.tabular'
    output_path = f'{root}/MBE/20260929/figures/Fig.fastani_clustered_heatmap.pdf'

    print("reading FastANI output file...")
    df = read_fastani_output(file_path)
    print(f"reading {len(df)} records")


    print("generating a matrix...")
    matrix = build_similarity_matrix(df)
    print(f"matrix shape: {matrix.shape}")

    g = plot_clustered_heatmap(
        matrix,
        title="Claviceps FastANI Genome Similarity Clustered Heatmap",
        output_file=output_path)

    matrix.to_csv('fastani_matrix.csv')
    print("similarity matrix deposited: fastani_matrix.csv")

    return matrix, g


if __name__ == "__main__":
    matrix, g = main()